In [12]:
import tqdm
import pickle
import warnings
import numpy as np
import scipy as sp
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from copy import deepcopy
from scipy.spatial import distance

from data import *
from model import *
from utils import *
from recourse_model import LAROAR, RecourseCost

warnings.filterwarnings('ignore')
# pd.set_option('display.max_colwidth', 0)

In [13]:
def append_hist(d, i, x_0, theta_0, x_r, theta_r):
    d['i'].append(i)
    d['x_0'].append(x_0)
    d['theta_0'].append(theta_0)
    d['x_r'].append(x_r)
    d['theta_r'].append(theta_r)

In [14]:
def recourse_model_runner(dataset: np.ndarray, model: LAROAR):    
    hist = {'i': [], 'x_0': [], 'theta_0': [], 'x_r': [], 'theta_r': []}
    n = len(dataset)
        
    for i in tqdm.trange(n, desc=f'Eval alpha={model.alpha}; lambda={model.lamb}', colour='#0091ff'):
        x_0 = dataset[i]
        theta_0 = np.hstack((model.weights, model.bias))

        x_r = model.get_recourse(x_0, beta=1.)
        weights_r, bias_r = model.calc_theta_adv(x_r)
        theta_r = np.hstack((weights_r, bias_r))
        
        append_hist(hist, i, x_0, theta_0, x_r, theta_r)
    
    df_hist = pd.DataFrame(hist)  
    return df_hist

In [15]:
def recourse_tradeoff(params: dict):
    if params['data'] in ['correction', 'german']:
        data_model = CorrectionShift("../datasets/german.csv", "../datasets/corrected_german.csv", seed=0)
    elif params['data'] in ['temporal', 'business']:
        data_model = TemporalShift("../datasets/SBAcase.11.13.17.csv", seed=0)
    elif params['data'] in ['geospatial', 'student']:
        data_model = GeospatialShift("../datasets/student-por.csv", seed=0)
    elif params['data'] in ['synthetic', 'simulated']:
        data_model = SyntheticData(alpha=1.5, beta=0, n=200, seed=0)
    
    data1, _ = data_model.get_data(0)
    X1_train, y1_train, X1_test, y1_test = data1

    base_model = LR()
        
    base_model.train(X1_train.values, y1_train.values)
    
    recourse_needed_X1_train = recourse_needed(base_model.predict, X1_train.values)
    recourse_needed_X1_test = recourse_needed(base_model.predict, X1_test.values)
    
    alpha = params['alpha']
    weights, bias = base_model.model.coef_[0], base_model.model.intercept_
        
    recourse_model = LAROAR(
        weights = weights,
        bias = bias,
        alpha = alpha,
    )    
    
    lamb = recourse_model.choose_lambda(recourse_needed_X1_train, base_model.predict, X1_train.values, base_model.predict_proba)
    recourse_model.lamb = lamb
        
    df_hist = recourse_model_runner(recourse_needed_X1_test, recourse_model)

    return df_hist

In [16]:
params = {}
# 'synthetic/simulated', 'correction/german', 'temporal/business', 'geospatial/student'
params['data'] = 'synthetic'
params['alpha'] = 0.5

df_hist = recourse_tradeoff(params)

Choosing lambda


8it [00:00, 241.79it/s]
Eval alpha=0.1; lambda=0.8: 100%|██████████| 21/21 [00:00<00:00, 15578.42it/s]


In [18]:
theta_rs = np.stack(df_hist.theta_r.values)
unique_theta_rs = np.unique(theta_rs, axis=0)
unique_theta_rs.round(2)

array([[1.61, 1.47, 0.07]])

In [20]:
theta_0s = np.stack(df_hist.theta_0.values)
unique_theta_0s = np.unique(theta_0s, axis=0)
unique_theta_0s.round(2)

array([[1.51, 1.57, 0.17]])

In [21]:
n = len(unique_theta_rs)
rng = np.random.default_rng(0)
idx = rng.choice(range(n), 2 if n>=2 else 1, replace=False)
two_unique_theta_rs = unique_theta_rs[idx]
two_unique_theta_rs.round(2)

array([[1.61, 1.47, 0.07]])

In [25]:
if params['data'] == 'synthetic':
    theta_r = deepcopy(two_unique_theta_rs[0])
    theta_r[0] = unique_theta_0s[0][0]

    two_unique_theta_rs = np.vstack((two_unique_theta_rs, theta_r))
    two_unique_theta_rs

array([1.51374515, 1.47217731, 0.07049114])

In [27]:
np.save(f'../theta_preds/theta_preds_{params["data"]}.npy', two_unique_theta_rs)

In [28]:
np.load(f'../theta_preds/theta_preds_{params["data"]}.npy').round(2)

array([[1.61, 1.47, 0.07],
       [1.51, 1.47, 0.07]])